# LSEG Data Pull - Macaulay Duration

## 0. Setup

In [11]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import lseg.data as ld

warnings.filterwarnings("ignore", category=FutureWarning, module="lseg")
pd.set_option("display.max_columns", 120)

BASE_DIR = Path("/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data")
DATA_DIR = BASE_DIR / "intermediate"
CACHE_DATA_DIR = BASE_DIR / "cache"
CACHE_DATA_DIR.mkdir(parents=True, exist_ok=True)

import hashlib
import json
import random
import re
import time
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]


## 1. Input-Parquets laden

In [12]:
EURO500_PATH = DATA_DIR / "euro500.parquet"
EURO500_RETURNS_PATH = DATA_DIR / "euro500_index_returns.parquet"
DAILY_RETURNS_IN_INDEX_PATH = DATA_DIR / "euro500_daily_returns.parquet"

for p in [EURO500_PATH, EURO500_RETURNS_PATH, DAILY_RETURNS_IN_INDEX_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")

euro500 = pd.read_parquet(EURO500_PATH)
euro500_returns = pd.read_parquet(EURO500_RETURNS_PATH)
daily_returns_euro500_in_index = pd.read_parquet(DAILY_RETURNS_IN_INDEX_PATH)

print("Loaded:")
print("- euro500:", euro500.shape)
print("- euro500_returns:", euro500_returns.shape)
print("- daily_returns_euro500_in_index:", daily_returns_euro500_in_index.shape)

Loaded:
- euro500: (56500, 25)
- euro500_returns: (7016, 8)
- daily_returns_euro500_in_index: (3457796, 10)


## 2. Shares Outstanding

Fehlende Werte in `shares_outstanding` werden imputiert mit:

$$
\text{shares\_outstanding} = \frac{\text{mcap\_eur}}{\text{PriceClose}}
$$



In [13]:
# Step 2 — Missing shares_outstanding mit mcap_eur / PriceClose auffuellen
required_cols = ['shares_outstanding', 'mcap_eur', 'PriceClose']
missing_required = [c for c in required_cols if c not in euro500.columns]
if missing_required:
    raise KeyError(f"Missing required columns for Step 2: {missing_required}")

for c in required_cols:
    euro500[c] = pd.to_numeric(euro500[c], errors='coerce')

n_total = len(euro500)
missing_before = int(euro500['shares_outstanding'].isna().sum())
coverage_before = 1.0 - (missing_before / n_total) if n_total else np.nan

# Nur dort imputen, wo shares_outstanding fehlt und Divisor valide ist.
fill_mask = (
    euro500['shares_outstanding'].isna()
    & euro500['mcap_eur'].notna()
    & euro500['PriceClose'].notna()
    & (euro500['PriceClose'] != 0)
)

euro500.loc[fill_mask, 'shares_outstanding'] = (
    euro500.loc[fill_mask, 'mcap_eur'] / euro500.loc[fill_mask, 'PriceClose']
)

missing_after = int(euro500['shares_outstanding'].isna().sum())
coverage_after = 1.0 - (missing_after / n_total) if n_total else np.nan
filled_rows = int(fill_mask.sum())

print('Step 2 coverage update (shares_outstanding):')
print(f'- total rows: {n_total}')
print(f'- missing before: {missing_before} ({(1-coverage_before)*100:.2f}%)')
print(f'- missing after : {missing_after} ({(1-coverage_after)*100:.2f}%)')
print(f'- filled rows   : {filled_rows}')
print(f'- coverage before: {coverage_before*100:.2f}%')
print(f'- coverage after : {coverage_after*100:.2f}%')

Step 2 coverage update (shares_outstanding):
- total rows: 56500
- missing before: 18025 (31.90%)
- missing after : 368 (0.65%)
- filled rows   : 17657
- coverage before: 68.10%
- coverage after : 99.35%


## 3. Cash EPS Forecasts + Act Value ziehen (`euro500_CashEPSAct`)

In [15]:
from typing import Dict, List, Optional, Tuple
import concurrent.futures as _cf
# ------------------------------------------------------------
# Step 3 — Cash EPS Forecasts FY1-FY5 + Current Act Value
# Cache: per company (firm_id), extensible across index changes
# Fixes vs. old code: no ISIN:-prefix, rate-limit backoff not abort
# ------------------------------------------------------------


def _asof_from_euro500(df: pd.DataFrame) -> pd.Series:
    if "date" not in df.columns:
        raise ValueError("Missing required date column in euro500.")
    return pd.to_datetime(df["date"], errors="coerce").dt.normalize()

# ---- Config ------------------------------------------------
HORIZONS          = ["FY1", "FY2", "FY3", "FY4", "FY5"]
FY_FIELDS_MAP_S3   = {f"EPS_{h}": f"TR.CashEPSMean(period={h})" for h in HORIZONS}
ACT_FIELD_MAP_S3   = {"EPS_CashAct": "TR.EPSCashActValue"}
EPS_FIELDS_MAP     = {**FY_FIELDS_MAP_S3, **ACT_FIELD_MAP_S3}
TARGET_COLS_S3     = list(EPS_FIELDS_MAP.keys())
REQUIRED_COLS_S3   = list(FY_FIELDS_MAP_S3.keys())

BATCH_SIZE_S3                = 100
FORCE_REFRESH_S3             = False
CACHE_ONLY_S3                = False
# Master switch: True => Step 3 never pulls from LSEG network, only local store/cache.
DISABLE_LSEG_PULL_S3         = False
MAX_RETRIES_S3               = 1
DEBUG_VERBOSE_S3             = False
RESUME_FROM_LAST_FILLED_S3   = True
INCLUDE_LAST_FILLED_S3      = False   # True => recompute the last filled date as well

# Effective offline mode (also respects existing CACHE_ONLY_S3 flag).
CACHE_ONLY_S3                = bool(CACHE_ONLY_S3 or DISABLE_LSEG_PULL_S3)

RATE_LIMIT_COOLDOWN_SEC_S3   = 5.0
RATE_LIMIT_MULTIPLIER_S3     = 2.0
RATE_LIMIT_HARD_PAUSE_SEC_S3 = 30.0
FAIL_FAST_ON_RATE_LIMIT_S3   = False   # allow retry on 429 (not abort)
S3_CALL_TIMEOUT_SEC          = 45.0    # hard timeout per ld.get_data call to avoid long silent hangs

CACHE_VERSION_S3 = "v4_cash_eps_mean_act"
EURO500_EPS_PATH = DATA_DIR / "euro500_CashEPSAct.parquet"
S3_ROWS_PATH     = CACHE_DATA_DIR / "cash_eps_act_step3_rows_v4.parquet"
S3_CKPT_PATH     = CACHE_DATA_DIR / "cash_eps_act_step3_checkpoint_v4.json"
S3_CACHE_DIR     = CACHE_DATA_DIR / "cash_eps_act_cache_by_company"
S3_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Suppress LSEG/pandas incompatibility: LSEG internals do `if df:` in __del__ callbacks,
# which raises ValueError in pandas 2.x. These are unraisable exceptions that bypass
# try/except and show as cell errors in Python 3.12+/Jupyter.
import sys as _sys
_s3_prev_unraisablehook = _sys.unraisablehook
def _s3_unraisable_hook(args):
    # LSEG destructors may trigger pandas bool(ValueError) in unraisable context.
    try:
        msg = str(getattr(args, "exc_value", "")).lower()
    except Exception:
        msg = ""
    if "truth value" in msg and "ambiguous" in msg and ("series" in msg or "dataframe" in msg):
        return  # suppress known non-fatal pandas truth-value errors
    _s3_prev_unraisablehook(args)
_sys.unraisablehook = _s3_unraisable_hook

# pandas 2.x + Python 3.13 compatibility: some third-party code does `if df:`
# (or `if series:`), which now raises ValueError. Map truthiness to not-empty.
def _s3_ndframe_bool(self):
    try:
        return not self.empty
    except Exception:
        return True

try:
    _s3_orig_series_bool = pd.Series.__bool__
except Exception:
    _s3_orig_series_bool = None
try:
    _s3_orig_series_nonzero = pd.Series.__nonzero__
except Exception:
    _s3_orig_series_nonzero = None
try:
    _s3_orig_df_bool = pd.DataFrame.__bool__
except Exception:
    _s3_orig_df_bool = None
try:
    _s3_orig_df_nonzero = pd.DataFrame.__nonzero__
except Exception:
    _s3_orig_df_nonzero = None

pd.Series.__bool__ = _s3_ndframe_bool
pd.Series.__nonzero__ = _s3_ndframe_bool
pd.DataFrame.__bool__ = _s3_ndframe_bool
pd.DataFrame.__nonzero__ = _s3_ndframe_bool

# ---- Basic helpers -----------------------------------------
def _s3_asof(df: pd.DataFrame) -> pd.Series:
    if "date" not in df.columns:
        raise ValueError("Missing required date column for Step 3.")
    return pd.to_datetime(df["date"], errors="coerce").dt.normalize()

def _s3_clean(s: pd.Series) -> pd.Series:
    x = s.astype("string").str.strip()
    return x.where(x.notna() & (x != ""), pd.NA)

def _s3_norm_isin(v) -> Optional[str]:
    if pd.isna(v):
        return None
    s = str(v).strip()
    return (s.split(":", 1)[1].strip() if s.upper().startswith("ISIN:") else s) or None

def _s3_build_company_candidates(df: pd.DataFrame) -> List[Tuple[str, str]]:
    """Ordered (id_type, pull_id) list for one company.
    Order: RIC_current → RIC → bare ISINs → pull_id history.
    RIC-first is more reliable for TR.CashEPSMean snapshots; ISIN remains fallback.
    Never adds ISIN:-prefixed variants."""
    q = df.sort_values("date")
    out, seen = [], set()

    def _add(id_type: str, value):
        if pd.isna(value):
            return
        v = str(value).strip()
        if not v:
            return
        key = (id_type.upper(), v)
        if key not in seen:
            seen.add(key)
            out.append(key)

    for val in q.get("RIC_current", pd.Series(dtype="object")):
        _add("RIC", val)
    for val in q.get("RIC", pd.Series(dtype="object")):
        _add("RIC", val)
    for val in q.get("ISIN", pd.Series(dtype="object")):
        n = _s3_norm_isin(val)
        if n:
            _add("ISIN", n)
    if "pull_id" in q.columns and "id_type" in q.columns:
        for id_type, pull_id in zip(q["id_type"], q["pull_id"]):
            it = str(id_type).upper().strip() if pd.notna(id_type) else ""
            if it == "ISIN":
                n = _s3_norm_isin(pull_id)
                if n:
                    _add("ISIN", n)
            elif it == "RIC":
                _add("RIC", pull_id)
    return out

# ---- Per-company cache (extensible across index changes) ---
_s3_cache_mem: Dict[str, pd.DataFrame] = {}

def _s3_cache_file(firm_id: str) -> Path:
    key = str(firm_id).upper().strip()
    h = hashlib.sha1(key.encode()).hexdigest()[:16]
    clean = re.sub(r"[^A-Za-z0-9._-]", "_", key[:60])
    return S3_CACHE_DIR / f"{clean}__{h}__{CACHE_VERSION_S3}.parquet"

def _s3_norm_cache(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x["date"] = pd.to_datetime(x.get("date"), errors="coerce").dt.normalize()
    for c in TARGET_COLS_S3:
        if c not in x.columns:
            x[c] = np.nan
        x[c] = pd.to_numeric(x[c], errors="coerce")
    x = x.dropna(subset=["date"]).drop_duplicates("date", keep="last").sort_values("date")
    return x[["date"] + TARGET_COLS_S3].reset_index(drop=True)

def _s3_get_cache(firm_id: str) -> pd.DataFrame:
    if firm_id not in _s3_cache_mem:
        p = _s3_cache_file(firm_id)
        try:
            _s3_cache_mem[firm_id] = _s3_norm_cache(pd.read_parquet(p)) if p.exists() else pd.DataFrame(columns=["date"] + TARGET_COLS_S3)
        except Exception:
            _s3_cache_mem[firm_id] = pd.DataFrame(columns=["date"] + TARGET_COLS_S3)
    return _s3_cache_mem[firm_id]

def _s3_put_cache(firm_id: str, df: pd.DataFrame) -> None:
    norm = _s3_norm_cache(df)
    _s3_cache_mem[firm_id] = norm
    p = _s3_cache_file(firm_id)
    tmp = p.with_suffix(p.suffix + ".tmp")
    norm.to_parquet(tmp, index=False)
    tmp.replace(p)

# ---- LSEG API call with rate-limit backoff -----------------
def _s3_call_get_data(universe: List[str], date, raise_on_error: bool = False) -> pd.DataFrame:
    """ld.get_data for EPS snapshot at date; retries on 429 with backoff."""
    asof_str = pd.to_datetime(date).strftime("%Y-%m-%d")
    cooldown = RATE_LIMIT_COOLDOWN_SEC_S3
    last_err = None
    for attempt in range(MAX_RETRIES_S3 + 1):
        try:
            ex = _cf.ThreadPoolExecutor(max_workers=1)
            try:
                fut = ex.submit(
                    ld.get_data,
                    universe=list(universe),
                    fields=list(EPS_FIELDS_MAP.values()),
                    parameters={"SDate": asof_str, "EDate": asof_str},
                )
                raw = fut.result(timeout=float(S3_CALL_TIMEOUT_SEC))
            finally:
                # Do not block on hung worker thread when timeout is hit.
                ex.shutdown(wait=False, cancel_futures=True)
            return pd.DataFrame(raw) if raw is not None else pd.DataFrame()
        except Exception as e:
            last_err = e
            if isinstance(e, TimeoutError):
                print(f"[WARN S3 timeout] attempt={attempt+1}/{MAX_RETRIES_S3+1} sec={S3_CALL_TIMEOUT_SEC:.0f} ids={len(universe)}", flush=True)
            msg = str(e).lower()
            is_rl = "too many requests" in msg or bool(re.search(r"\b429\b", msg))
            if is_rl:
                if FAIL_FAST_ON_RATE_LIMIT_S3:
                    raise
                pause = min(cooldown, RATE_LIMIT_HARD_PAUSE_SEC_S3)
                print(f"[WARN S3 rate-limit] attempt={attempt+1}/{MAX_RETRIES_S3+1} sleep={pause:.0f}s | {e}", flush=True)
                time.sleep(pause)
                cooldown *= RATE_LIMIT_MULTIPLIER_S3
                continue
            if attempt >= MAX_RETRIES_S3:
                break
            time.sleep(1.0 * (2 ** attempt))
    if last_err is not None:
        # In recursive isolation mode (raise_on_error=True), failures are expected
        # and handled by caller via split strategy; avoid warning spam.
        if not raise_on_error:
            print(f"[WARN S3 get_data failed] {type(last_err).__name__}: {last_err}", flush=True)
        if raise_on_error:
            raise last_err
    return pd.DataFrame()

# ---- Response parser ---------------------------------------
def _s3_num_scalar(v, prefer_idx: int | None = None):
    """Normalize arbitrary LSEG cell payloads to a numeric scalar.

    If payload is sequence-like, prefer_idx can select the FY-position instead of
    always taking the first element.
    """
    if isinstance(v, pd.DataFrame):
        v = v.stack(dropna=True)

    if isinstance(v, pd.Series):
        s = v.dropna()
        if s.empty:
            return np.nan
        if prefer_idx is not None and 0 <= int(prefer_idx) < len(s):
            v = s.iloc[int(prefer_idx)]
        else:
            v = s.iloc[0]
    elif isinstance(v, (list, tuple, np.ndarray)):
        seq = pd.Series(list(v)).dropna()
        if seq.empty:
            return np.nan
        if prefer_idx is not None and 0 <= int(prefer_idx) < len(seq):
            v = seq.iloc[int(prefer_idx)]
        else:
            v = seq.iloc[0]

    return pd.to_numeric(v, errors="coerce")

def _s3_parse_response(raw_df: pd.DataFrame, ids: List[str]) -> Dict[str, Dict]:
    """Parse LSEG response -> {pull_id_upper: {EPS_FY1: float, ...}}.
    Columns assigned by position (same order as EPS_FIELDS_MAP).
    Instrument matching: exact -> contains -> single-id fallback."""
    if raw_df is None or raw_df.empty:
        return {}
    x = raw_df.copy()
    if isinstance(x.columns, pd.MultiIndex):
        x.columns = [" | ".join(str(v) for v in tup if v).strip() for tup in x.columns]
    else:
        x.columns = [str(c).strip() for c in x.columns]

    # Identify instrument column
    inst_col = x.columns[0]
    for c in x.columns:
        if c.lower() in {"instrument", "ric", "isin"} or "instrument" in c.lower():
            inst_col = c
            break

    inst_pos    = list(x.columns).index(inst_col)
    value_pos   = [j for j in range(len(x.columns)) if j != inst_pos]
    ids_norm    = {str(i).strip().upper(): str(i).strip() for i in ids}
    inst_series = x[inst_col].astype("string").str.strip().str.upper()
    out: Dict[str, Dict] = {}

    for row_idx in range(len(x)):
        inst = inst_series.iloc[row_idx]
        if pd.isna(inst):
            continue
        # 1) Exact match
        matched = ids_norm.get(inst)
        # 2) Contains fallback (LSEG sometimes strips/decorates identifiers)
        if matched is None:
            for norm_id in ids_norm:
                if re.search(re.escape(norm_id), inst, re.IGNORECASE) or \
                   re.search(re.escape(inst), norm_id, re.IGNORECASE):
                    matched = ids_norm[norm_id]
                    break
        # 3) Single-id fallback
        if matched is None and len(ids) == 1:
            matched = list(ids_norm.values())[0]
        if matched is None:
            continue

        values = {}
        for i, c in enumerate(TARGET_COLS_S3):
            if i < len(value_pos):
                # Positional access avoids duplicate-column-label collisions in LSEG responses.
                raw_cell = x.iloc[row_idx, value_pos[i]]
                values[c] = _s3_num_scalar(raw_cell, prefer_idx=i)
            else:
                values[c] = np.nan
        out[matched.strip().upper()] = values

    return out

_s3_last_fetch_error: str = ""

def _s3_fetch_parsed_recursive(ids: List[str], date, max_depth: int = 10) -> Tuple[Dict[str, Dict], List[str]]:
    """Fetch EPS robustly: split failing groups to isolate bad identifiers."""
    ids = [str(i).strip() for i in ids if str(i).strip()]
    if not ids:
        return {}, []

    global _s3_last_fetch_error
    try:
        raw = _s3_call_get_data(ids, date, raise_on_error=True)
    except Exception as e:
        _s3_last_fetch_error = f"{type(e).__name__}: {e}"
        if len(ids) == 1 or max_depth <= 0:
            return {}, ids[:1]
        mid = max(1, len(ids) // 2)
        p_l, b_l = _s3_fetch_parsed_recursive(ids[:mid], date, max_depth=max_depth - 1)
        p_r, b_r = _s3_fetch_parsed_recursive(ids[mid:], date, max_depth=max_depth - 1)
        out = {}
        out.update(p_l)
        out.update(p_r)
        return out, (b_l + b_r)

    if raw is None or raw.empty:
        return {}, []

    parsed = _s3_parse_response(raw, ids)
    if parsed:
        return parsed, []

    if len(ids) == 1 or max_depth <= 0:
        return {}, []
    mid = max(1, len(ids) // 2)
    p_l, b_l = _s3_fetch_parsed_recursive(ids[:mid], date, max_depth=max_depth - 1)
    p_r, b_r = _s3_fetch_parsed_recursive(ids[mid:], date, max_depth=max_depth - 1)
    out = {}
    out.update(p_l)
    out.update(p_r)
    return out, (b_l + b_r)


# ---- Store management --------------------------------------
_S3_PANEL_COLS = ["firm_id", "date"] + TARGET_COLS_S3

def _s3_prep_store(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=_S3_PANEL_COLS)
    x = df.copy()
    for c in _S3_PANEL_COLS:
        if c not in x.columns:
            x[c] = np.nan if c in TARGET_COLS_S3 else pd.NA
    x["date"] = pd.to_datetime(x["date"], errors="coerce").dt.normalize()
    for c in TARGET_COLS_S3:
        x[c] = pd.to_numeric(x[c], errors="coerce")
    return (x[_S3_PANEL_COLS]
            .dropna(subset=["firm_id", "date"])
            .drop_duplicates(["firm_id", "date"], keep="last")
            .reset_index(drop=True))

def _s3_upsert_store(store: pd.DataFrame, new_rows: pd.DataFrame) -> pd.DataFrame:
    s, n = _s3_prep_store(store), _s3_prep_store(new_rows)
    if n.empty:
        return s
    if s.empty:
        return n
    keep = s.merge(n[["firm_id", "date"]], on=["firm_id", "date"],
                   how="left", indicator=True)
    keep = keep[keep["_merge"] == "left_only"].drop(columns=["_merge"])
    return _s3_prep_store(pd.concat([keep, n], ignore_index=True))

# ---- Build request table -----------------------------------
euro500_eps = euro500.copy()
euro500_eps["date"] = _s3_asof(euro500_eps)
for c in ("ISIN", "RIC_current", "RIC", "pull_id", "id_type", "firm_id"):
    if c in euro500_eps.columns:
        euro500_eps[c] = _s3_clean(euro500_eps[c])

if "firm_id" not in euro500_eps.columns or not euro500_eps["firm_id"].notna().any():
    _fid = _s3_clean(euro500_eps.get("ISIN",        pd.Series(pd.NA, index=euro500_eps.index, dtype="string")))
    _fid = _fid.fillna(_s3_clean(euro500_eps.get("RIC_current", pd.Series(pd.NA, index=euro500_eps.index, dtype="string"))))
    _fid = _fid.fillna(_s3_clean(euro500_eps.get("RIC",         pd.Series(pd.NA, index=euro500_eps.index, dtype="string"))))
    euro500_eps["firm_id"] = "CID:" + _fid.astype("string")
else:
    euro500_eps["firm_id"] = _s3_clean(euro500_eps["firm_id"])

_req_cols = [c for c in ["firm_id", "date", "ISIN", "RIC_current", "RIC", "pull_id", "id_type"]
             if c in euro500_eps.columns]
req_all = (euro500_eps[_req_cols]
           .dropna(subset=["firm_id", "date"])
           .drop_duplicates()
           .reset_index(drop=True))

if req_all.empty:
    raise ValueError("No valid (firm_id, date) rows found for Step 3.")

# Per-company candidate maps
cand_map = {str(fid): _s3_build_company_candidates(g)
            for fid, g in req_all.groupby("firm_id", sort=False)}

# Flat candidate table  (firm_id x date x rank)
_cand_rows = []
for fid, g in req_all.groupby("firm_id", sort=False):
    cands = cand_map.get(str(fid), [])
    if not cands:
        continue
    for asof in g["date"].dropna().dt.normalize().unique():
        for rank, (id_type, pull_id) in enumerate(cands, start=1):
            _cand_rows.append({"firm_id": str(fid),
                                "date": pd.to_datetime(asof).normalize(),
                                "rank": rank, "id_type": id_type, "pull_id": str(pull_id)})

cand_df  = (pd.DataFrame(_cand_rows) if _cand_rows
            else pd.DataFrame(columns=["firm_id", "date", "rank", "id_type", "pull_id"]))
max_rank = int(cand_df["rank"].max()) if not cand_df.empty else 0

print(f"Step 3 -- firm x asof rows: {len(req_all)}")
print(f"Firms: {req_all['firm_id'].nunique()}  |  "
      f"asof range: {req_all['date'].min().date()} -> {req_all['date'].max().date()}")
print(f"Max ID-fallback ranks: {max_rank}  |  "
      f"Mode: {'CACHE_ONLY (LSEG disabled)' if DISABLE_LSEG_PULL_S3 else ('CACHE_ONLY' if CACHE_ONLY_S3 else 'CACHE+NETWORK')}")
if cand_df.empty:
    raise ValueError("No candidate identifiers.")

# ---- Load existing store & build working panel -------------
s3_store = pd.DataFrame(columns=_S3_PANEL_COLS)
if S3_ROWS_PATH.exists() and not FORCE_REFRESH_S3:
    try:
        s3_store = _s3_prep_store(pd.read_parquet(S3_ROWS_PATH))
        print(f"Loaded S3 store: {len(s3_store)} rows ({s3_store['firm_id'].nunique()} firms)")
    except Exception as e:
        print(f"[WARN] Could not load S3 store: {e}")

panel = req_all[["firm_id", "date"]].drop_duplicates().copy()
for c in TARGET_COLS_S3:
    panel[c] = np.nan

if not s3_store.empty and not FORCE_REFRESH_S3:
    panel = panel.merge(s3_store[["firm_id", "date"] + TARGET_COLS_S3],
                        on=["firm_id", "date"], how="left", suffixes=("", "_old"))
    for c in TARGET_COLS_S3:
        panel[c] = (pd.to_numeric(panel[f"{c}_old"], errors="coerce")
                    .combine_first(pd.to_numeric(panel[c], errors="coerce")))
    panel = panel.drop(columns=[c for c in panel.columns if c.endswith("_old")])

pre_filled = int(panel[REQUIRED_COLS_S3].notna().all(axis=1).sum())
print(f"Pre-filled from store: {pre_filled}/{len(panel)} rows already complete")

# ---- Main pull loop ----------------------------------------
run_t0        = time.time()
progress_rows = []

if not CACHE_ONLY_S3:
    ld.open_session()

try:
    dates_all = sorted(panel["date"].dropna().unique().tolist())
    dates = dates_all

    if RESUME_FROM_LAST_FILLED_S3 and (not FORCE_REFRESH_S3):
        per_row_any = panel[REQUIRED_COLS_S3].notna().any(axis=1)
        per_date_any = per_row_any.groupby(panel["date"]).any()
        filled_dates = sorted(pd.to_datetime(per_date_any[per_date_any].index, errors="coerce"))
        if filled_dates:
            last_filled = pd.to_datetime(filled_dates[-1]).normalize()
            if INCLUDE_LAST_FILLED_S3:
                dates = [d for d in dates_all if pd.to_datetime(d).normalize() >= last_filled]
                print(f"[S3 resume] last quarter with EPS values: {last_filled.date()} -> include this date", flush=True)
            else:
                dates = [d for d in dates_all if pd.to_datetime(d).normalize() > last_filled]
                print(f"[S3 resume] last quarter with EPS values: {last_filled.date()} -> start from next date", flush=True)

    if not dates:
        print("[S3] No pending asof dates after resume filter.", flush=True)

    print(f"\n[S3] Pull start: {len(dates)} asof-dates, max_rank={max_rank}", flush=True)

    for d_ix, asof in enumerate(dates, 1):
        asof_ts   = pd.to_datetime(asof).normalize()
        date_mask = panel["date"] == asof_ts
        n_total   = int(date_mask.sum())
        print(f"\n[S3] asof {d_ix}/{len(dates)} = {asof_ts.date()}  rows={n_total}", flush=True)

        for rank in range(1, max_rank + 1):
            unres_mask = date_mask & panel[REQUIRED_COLS_S3].isna().any(axis=1)
            n_unres    = int(unres_mask.sum())
            if n_unres == 0:
                print(f"  rank={rank}: fully resolved -> skip", flush=True)
                break

            # Candidates for this rank / date / unresolved firms
            unres_firms = set(panel.loc[unres_mask, "firm_id"].astype(str))
            rank_cands  = cand_df[(cand_df["date"] == asof_ts) &
                                  (cand_df["rank"] == rank) &
                                  (cand_df["firm_id"].astype(str).isin(unres_firms))].copy()
            if rank_cands.empty:
                continue

            # pid -> [firm_ids] map (handles multiple firms sharing same ISIN/RIC)
            pid_to_firms: Dict[str, List[str]] = {}
            for _, row in rank_cands.iterrows():
                pid_key = str(row["pull_id"]).strip().upper()
                pid_to_firms.setdefault(pid_key, []).append(str(row["firm_id"]))

            # Check per-company caches first (exact date only)
            pids_need_network: List[str] = []
            cache_hit_count = 0
            for _, row in rank_cands.iterrows():
                fid = str(row["firm_id"])
                pid = str(row["pull_id"]).strip().upper()
                cached = _s3_get_cache(fid)
                if not cached.empty:
                    hit = cached[cached["date"] == asof_ts]
                    if not hit.empty and hit[REQUIRED_COLS_S3].notna().values.any():
                        m = (panel["firm_id"] == fid) & (panel["date"] == asof_ts)
                        for c in TARGET_COLS_S3:
                            v = _s3_num_scalar(hit[c].iloc[-1])
                            if pd.notna(v):
                                panel.loc[m & panel[c].isna(), c] = v
                        cache_hit_count += 1
                        continue
                pids_need_network.append(pid)

            pids_need_network = sorted(set(pids_need_network))
            print(f"  rank={rank}: unresolved={n_unres} "
                  f"cache_hits={cache_hit_count} network_pids={len(pids_need_network)}", flush=True)

            # Network pull
            if pids_need_network and not CACHE_ONLY_S3:
                chunks = [pids_need_network[i:i + BATCH_SIZE_S3]
                          for i in range(0, len(pids_need_network), BATCH_SIZE_S3)]
                for ch_ix, chunk in enumerate(chunks, 1):
                    print(f"    chunk {ch_ix}/{len(chunks)} ids={len(chunk)} start", flush=True)
                    t0 = time.time()
                    parsed, bad_ids = _s3_fetch_parsed_recursive(chunk, asof_ts, max_depth=10)
                    dt = time.time() - t0
                    print(f"    chunk {ch_ix}/{len(chunks)} ids={len(chunk)} got={len(parsed)} bad={len(bad_ids)} sec={dt:.1f}", flush=True)

                    for pid_upper, values in parsed.items():
                        for fid in pid_to_firms.get(pid_upper, []):
                            # Update working panel (fill NaNs only)
                            m = (panel["firm_id"] == fid) & (panel["date"] == asof_ts)
                            for c in TARGET_COLS_S3:
                                v = _s3_num_scalar(values.get(c))
                                if pd.notna(v):
                                    panel.loc[m & panel[c].isna(), c] = v
                            # Persist to per-company cache
                            existing = _s3_get_cache(fid)
                            new_row  = pd.DataFrame([{"date": asof_ts,
                                                      **{c: values.get(c, np.nan) for c in TARGET_COLS_S3}}])
                            if existing.empty:
                                _s3_put_cache(fid, new_row)
                            else:
                                _s3_put_cache(fid, pd.concat([existing, new_row], ignore_index=True))

                    if bad_ids:
                        show = ', '.join(bad_ids[:5])
                        more = '' if len(bad_ids) <= 5 else f' (+{len(bad_ids)-5} more)'
                        print(f"      unresolved identifiers in chunk: [{show}]{more}", flush=True)
                        if len(parsed) == 0 and len(bad_ids) == len(chunk):
                            err_hint = _s3_last_fetch_error if str(_s3_last_fetch_error).strip() else "unknown"
                            print(f"      chunk failure reason (last): {err_hint}", flush=True)

            elif pids_need_network and CACHE_ONLY_S3:
                print(f"    CACHE_ONLY_S3=True, skipping {len(pids_need_network)} network ids", flush=True)

        # ---- Progress for this date ----------------------------
        found    = {c: int(panel.loc[date_mask, c].notna().sum()) for c in TARGET_COLS_S3}
        resolved = int(panel.loc[date_mask, REQUIRED_COLS_S3].notna().all(axis=1).sum())
        elapsed  = time.time() - run_t0
        q_label  = f"{asof_ts.year}Q{asof_ts.quarter}"
        found_str = "  ".join(f"{c}={found[c]}/{n_total}" for c in TARGET_COLS_S3)
        print(f"[S3] {q_label} ({asof_ts.date()}) resolved={resolved}/{n_total} | "
              f"{found_str} | elapsed={elapsed/60:.1f}m", flush=True)

        progress_rows.append({"quarter": q_label,
                               "date": asof_ts.date().isoformat(),
                               "rows": n_total, "resolved": resolved,
                               "elapsed_min": round(elapsed / 60, 1),
                               **{f"found_{c}": found[c] for c in TARGET_COLS_S3}})

        # ---- Incremental save ----------------------------------
        s3_store = _s3_upsert_store(s3_store, panel.loc[date_mask].copy())
        _tmp = S3_ROWS_PATH.with_suffix(S3_ROWS_PATH.suffix + ".tmp")
        s3_store.to_parquet(_tmp, index=False)
        _tmp.replace(S3_ROWS_PATH)
        S3_CKPT_PATH.write_text(json.dumps({
            "updated_at": pd.Timestamp.utcnow().isoformat(),
            "last_date":  asof_ts.date().isoformat(),
            "store_rows": len(s3_store),
        }, indent=2))

finally:
    pass  # session left open intentionally; ld.close_session() can trigger
          # pandas-truthiness issues in some LSEG internals on pandas 2.x / Python 3.13

# ---- Merge results & save output ---------------------------
out = _s3_prep_store(panel)

# Deterministic write: do not reuse legacy EPS values from older runs.
# This prevents stale/misaligned history from leaking into the final table.
euro500_eps = euro500_eps.drop(columns=[c for c in TARGET_COLS_S3 if c in euro500_eps.columns], errors="ignore")

euro500_eps = euro500_eps.merge(
    out[["firm_id", "date"] + TARGET_COLS_S3],
    on=["firm_id", "date"],
    how="left",
    validate="m:1",
)

for c in TARGET_COLS_S3:
    euro500_eps[c] = pd.to_numeric(euro500_eps[c], errors="coerce")

euro500_eps.to_parquet(EURO500_EPS_PATH, index=False)

# ---- Summary -----------------------------------------------
if progress_rows:
    display(pd.DataFrame(progress_rows))
print(f"\nSaved: {EURO500_EPS_PATH}  rows={len(euro500_eps)}")
# Step 3 parser sanity: EPS horizons should not be identically equal for almost all rows
_eps_cols_chk = [f"EPS_{h}" for h in HORIZONS if f"EPS_{h}" in euro500_eps.columns]
if len(_eps_cols_chk) >= 2:
    _eps_num = euro500_eps[_eps_cols_chk].apply(pd.to_numeric, errors='coerce')
    _same_all = _eps_num.notna().all(axis=1) & (_eps_num.nunique(axis=1) == 1)
    _all_nonnull = int(_eps_num.notna().all(axis=1).sum())
    _same_cnt = int(_same_all.sum())
    print(f"Step 3 parser sanity: rows(all FY equal among non-null rows) = {_same_cnt} / {_all_nonnull}")
    if _all_nonnull > 0 and (_same_cnt / _all_nonnull) > 0.95:
        print("WARNING: EPS FY horizons look suspiciously identical (>95% equal across FY1-5). Check raw mapping.")

print(f"S3 store: {S3_ROWS_PATH}  rows={len(s3_store)}")
print(f"Cache dir: {S3_CACHE_DIR}")
for h in HORIZONS:
    c = f"EPS_{h}"
    if c in euro500_eps.columns:
        filled = int(euro500_eps[c].notna().sum())
        total  = len(euro500_eps)
        print(f"  {c}: {filled}/{total} filled ({100*filled/total:.1f}%)")












Step 3 -- firm x asof rows: 56500
Firms: 1248  |  asof range: 1997-12-31 -> 2025-12-31
Max ID-fallback ranks: 8  |  Mode: CACHE+NETWORK
Loaded S3 store: 500 rows (500 firms)
Pre-filled from store: 0/56500 rows already complete

[S3] Pull start: 113 asof-dates, max_rank=8

[S3] asof 1/113 = 1997-12-31  rows=500
  rank=1: unresolved=500 cache_hits=0 network_pids=500
    chunk 1/5 ids=100 start
[WARN S3 rate-limit] attempt=1/2 sleep=5s | Too many requests, please try again later. Requested universes: ['AAHG.F', 'AALB.AS', 'ABGV.VI', 'ABHA.MU', 'ABKG.MU', 'ACCP.PA', 'ACKB.BR', 'ACS.MC', 'ACX.MC', 'AD.AS', 'ADSGN.DE', 'ADZ.MC', 'AEDES.MI', 'AEGN.AS', 'AGRV.VI', 'AIBG.I', 'AIRF.PA', 'AIRP.PA', 'AKW.PA', 'AKZO.AS', 'ALATA.PA', 'ALATI.PA', 'ALAVI.PA', 'ALBAV.HE', 'ALBI.PA', 'ALBON.PA', 'ALBOU.PA', 'ALCBX.PA', 'ALCE.SCT', 'ALCGM.PA', 'ALCLA.PA', 'ALCOF.PA', 'ALCOI.PA', 'ALDEL.PA', 'ALDEV.PA', 'ALDV.PA', 'ALEXA.PA', 'ALGEV.PA', 'ALGIL.PA', 'ALGIR.PA', 'ALHF.PA', 'ALHRG.PA', 'ALLAN.PA', 'ALLEX.PA

KeyboardInterrupt: 

### 3B. Coverage-Analyse EPS (`EPS_FY1`-`EPS_FY5`)

In [ ]:
if not EURO500_EPS_PATH.exists():
    raise FileNotFoundError(f'Missing file: {EURO500_EPS_PATH}')

eps_cov = pd.read_parquet(EURO500_EPS_PATH).copy()

if 'date' not in eps_cov.columns:
    raise ValueError('Missing required date column for EPS coverage analysis.')
eps_cov['date'] = pd.to_datetime(eps_cov['date'], errors='coerce').dt.normalize()

eps_cov = eps_cov[eps_cov['date'].notna()].copy()
if eps_cov.empty:
    raise ValueError('No valid date values found for EPS coverage analysis.')

eps_cov['quarter'] = (eps_cov['date'] + pd.Timedelta(days=1)).dt.to_period('Q')
eps_cols = [c for c in ['EPS_FY1', 'EPS_FY2', 'EPS_FY3', 'EPS_FY4', 'EPS_FY5']]

for c in eps_cols:
    if c not in eps_cov.columns:
        eps_cov[c] = np.nan
    eps_cov[c] = pd.to_numeric(eps_cov[c], errors='coerce')

eps_cov['eps_any'] = eps_cov[eps_cols].notna().any(axis=1)
eps_cov['eps_all'] = eps_cov[eps_cols].notna().all(axis=1)
eps_cov['eps_partial'] = eps_cov['eps_any'] & (~eps_cov['eps_all'])
eps_cov['eps_non_null_n'] = eps_cov[eps_cols].notna().sum(axis=1)

agg_map = {
    'rows': ('quarter', 'size'),
    'eps_any_coverage': ('eps_any', 'mean'),
    'eps_all_coverage': ('eps_all', 'mean'),
    'eps_partial_coverage': ('eps_partial', 'mean'),
}
for c in eps_cols:
    agg_map[f'{c}_coverage'] = (c, lambda s: s.notna().mean())

quarterly_eps_cov = (
    eps_cov.groupby('quarter', as_index=False)
    .agg(**agg_map)
    .sort_values('quarter')
)

cov_cols = [c for c in quarterly_eps_cov.columns if c.endswith('_coverage')]
quarterly_eps_cov[cov_cols] = (quarterly_eps_cov[cov_cols] * 100.0).round(2)
quarterly_eps_cov['quarter'] = quarterly_eps_cov['quarter'].astype(str)

focus_eps_cov = quarterly_eps_cov.tail(8).copy()

summary_eps = pd.DataFrame([
    {
        'rows_total': int(len(eps_cov)),
        'asof_min': pd.to_datetime(eps_cov['date']).min(),
        'asof_max': pd.to_datetime(eps_cov['date']).max(),
        'unique_pull_ric': int(eps_cov['pull_ric'].nunique(dropna=True)) if 'pull_ric' in eps_cov.columns else np.nan,
        'eps_any_coverage_pct': round(float(eps_cov['eps_any'].mean() * 100.0), 2),
        'eps_all_coverage_pct': round(float(eps_cov['eps_all'].mean() * 100.0), 2),
        'eps_partial_coverage_pct': round(float(eps_cov['eps_partial'].mean() * 100.0), 2),
    }
])

print('EPS coverage summary:')
display(summary_eps)
print('EPS quarterly coverage (%):')
display(quarterly_eps_cov)


# Plot: quarterly EPS coverage per horizon (FY1-FY5)
fy_cov_cols = [f'{c}_coverage' for c in eps_cols if f'{c}_coverage' in quarterly_eps_cov.columns]
if fy_cov_cols:
    fig, ax = plt.subplots(figsize=(12, 5))
    x = quarterly_eps_cov['quarter'].astype(str).tolist()
    x_idx = np.arange(len(x))
    q1_idx = [i for i, q in enumerate(x) if q.endswith('Q1')]
    q1_labels = [x[i] for i in q1_idx]
    for col in fy_cov_cols:
        ax.plot(x_idx, quarterly_eps_cov[col], marker='o', label=col)
    ax.set_title('EPS quarterly coverage by horizon (Step 3B)')
    ax.set_xlabel('quarter')
    ax.set_ylabel('coverage (%)')
    ax.set_ylim(0, 100)
    ax.set_xticks(q1_idx)
    ax.set_xticklabels(q1_labels, rotation=45, ha='right')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No EPS_FY*_coverage columns available for Step 3B plot.')

print('EPS latest 8 quarters coverage (%):')
display(focus_eps_cov)

print('Rows with partial EPS coverage (any but not all horizons):', int(eps_cov['eps_partial'].sum()))

print('Rows with eps_any != eps_all:', int((eps_cov['eps_any'] != eps_cov['eps_all']).sum()))







## 4. Long-Term Growth (LTG) ergaenzen

In [ ]:
from typing import List, Tuple
# ------------------------------------------------------------
# Step 4 — Robust Long-Term Growth Pull (TR.LTGMean)
#          DailyReturns-style cache + ID fallback
# ------------------------------------------------------------



def _clean_str(s: pd.Series) -> pd.Series:
    x = s.astype('string').str.strip()
    return x.where(x.notna() & (x != ''), pd.NA)

LTG_FIELD = 'TR.LTGMean'
LTG_COL = LTG_FIELD
FORCE_REFRESH_LTG = True
LTG_MAX_RETRIES = 3
LTG_BASE_SLEEP = 0.4
LTG_ASOF_TOL_DAYS = 120
LTG_CACHE_ONLY = False  # False => allow LSEG pull (cache still used)
LTG_SKIP_FILLED = False
BATCH_SIZE_LTG = 100
BATCH_PAUSE_SEC_LTG = 0
LTG_ERR_PRINT_MAX = 8
LTG_ERR_PRINTED = 0

LTG_BAD_RICS_LOG = CACHE_DATA_DIR / 'ltg_bad_rics.csv'
LTG_STEP4_ROWS_PATH = CACHE_DATA_DIR / 'LTG_step4_rows.parquet'
LTG_STEP4_ROWS_LEGACY_PATH = CACHE_DATA_DIR / 'EPS_step4_rows.parquet'
LTG_STEP4_ROWS_READ_PATH = LTG_STEP4_ROWS_PATH if LTG_STEP4_ROWS_PATH.exists() else LTG_STEP4_ROWS_LEGACY_PATH
LTG_STEP4_CKPT_PATH = CACHE_DATA_DIR / 'LTG_step4_checkpoint.json'
LTG_STEP4_CKPT_LEGACY_PATH = CACHE_DATA_DIR / 'EPS_step4_checkpoint.json'
LTG_CACHE_DIR = CACHE_DATA_DIR / 'ltg_cache_by_company'
LTG_CACHE_DIR.mkdir(parents=True, exist_ok=True)

if not EURO500_EPS_PATH.exists():
    raise FileNotFoundError(f'Missing file: {EURO500_EPS_PATH}')

euro500_eps_ltg = pd.read_parquet(EURO500_EPS_PATH).copy()

# Keep prior LTG for explicit before-vs-after coverage comparison.
prev_ltg = pd.to_numeric(euro500_eps_ltg[LTG_COL], errors='coerce') if LTG_COL in euro500_eps_ltg.columns else pd.Series(np.nan, index=euro500_eps_ltg.index)

if 'date' not in euro500_eps_ltg.columns:
    raise ValueError('Missing required date column for Step 4 LTG pull.')
euro500_eps_ltg['date'] = pd.to_datetime(euro500_eps_ltg['date'], errors='coerce').dt.normalize()

for c in ['firm_id', 'ISIN', 'RIC_current', 'RIC', 'pull_ric', 'pull_id', 'id_type']:
    if c in euro500_eps_ltg.columns:
        euro500_eps_ltg[c] = _clean_str(euro500_eps_ltg[c])

# Fallback pull_id/id_type if missing.
if 'id_type' not in euro500_eps_ltg.columns:
    euro500_eps_ltg['id_type'] = np.select(
        [euro500_eps_ltg.get('ISIN', pd.Series(pd.NA, index=euro500_eps_ltg.index)).notna(),
         euro500_eps_ltg.get('RIC_current', pd.Series(pd.NA, index=euro500_eps_ltg.index)).notna(),
         euro500_eps_ltg.get('RIC', pd.Series(pd.NA, index=euro500_eps_ltg.index)).notna()],
        ['ISIN', 'RIC', 'RIC'],
        default=pd.NA,
    )
if 'pull_id' not in euro500_eps_ltg.columns:
    euro500_eps_ltg['pull_id'] = np.select(
        [euro500_eps_ltg.get('ISIN', pd.Series(pd.NA, index=euro500_eps_ltg.index)).notna(),
         euro500_eps_ltg.get('RIC_current', pd.Series(pd.NA, index=euro500_eps_ltg.index)).notna(),
         euro500_eps_ltg.get('RIC', pd.Series(pd.NA, index=euro500_eps_ltg.index)).notna()],
        [euro500_eps_ltg.get('ISIN', pd.Series(pd.NA, index=euro500_eps_ltg.index)),
         euro500_eps_ltg.get('RIC_current', pd.Series(pd.NA, index=euro500_eps_ltg.index)),
         euro500_eps_ltg.get('RIC', pd.Series(pd.NA, index=euro500_eps_ltg.index))],
        default=pd.NA,
    )

euro500_eps_ltg['id_type'] = _clean_str(euro500_eps_ltg['id_type'])
euro500_eps_ltg['pull_id'] = _clean_str(euro500_eps_ltg['pull_id'])

# Stable company key (close to DailyReturns logic).
if 'firm_id' not in euro500_eps_ltg.columns:
    if 'firm_id' in euro500_eps_ltg.columns and euro500_eps_ltg['firm_id'].notna().any():
        euro500_eps_ltg['firm_id'] = _clean_str(euro500_eps_ltg['firm_id'])
    else:
        euro500_eps_ltg['firm_id'] = _clean_str(euro500_eps_ltg.get('ISIN', pd.Series(pd.NA, index=euro500_eps_ltg.index, dtype='string')))
        euro500_eps_ltg['firm_id'] = euro500_eps_ltg['firm_id'].fillna(_clean_str(euro500_eps_ltg.get('RIC_current', pd.Series(pd.NA, index=euro500_eps_ltg.index, dtype='string'))))
        euro500_eps_ltg['firm_id'] = euro500_eps_ltg['firm_id'].fillna(_clean_str(euro500_eps_ltg.get('RIC', pd.Series(pd.NA, index=euro500_eps_ltg.index, dtype='string'))))
        euro500_eps_ltg['firm_id'] = 'CID:' + euro500_eps_ltg['firm_id'].astype('string')
else:
    euro500_eps_ltg['firm_id'] = _clean_str(euro500_eps_ltg['firm_id'])


def _safe_name(firm_id: str) -> str:
    h = hashlib.sha1(firm_id.encode('utf-8')).hexdigest()[:12]
    clean = re.sub(r'[^A-Za-z0-9._-]', '_', firm_id)
    return f"{clean[:80]}__{h}.parquet"


def _cache_path_for_company_id(firm_id: str, id_type: str, pull_id: str) -> Path:
    raw = f"{firm_id}|{id_type}|{pull_id}"
    h = hashlib.sha1(raw.encode('utf-8')).hexdigest()[:10]
    base = _safe_name(firm_id).replace('.parquet', '')
    suffix = re.sub(r'[^A-Za-z0-9._-]', '_', f"{id_type}_{h}")
    return LTG_CACHE_DIR / f"{base}__{suffix}.parquet"


def _cache_path_for_company_legacy(firm_id: str) -> Path:
    # Backward-compatible path used by older Step-4 versions.
    return LTG_CACHE_DIR / _safe_name(firm_id)


def _load_ltg_cache(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=['date', LTG_COL])
    d = pd.read_parquet(path).copy()
    if 'date' in d.columns:
        d['date'] = pd.to_datetime(d['date'], errors='coerce').dt.normalize()
    else:
        d['date'] = pd.NaT
    if LTG_COL not in d.columns:
        d[LTG_COL] = np.nan
    d[LTG_COL] = pd.to_numeric(d[LTG_COL], errors='coerce')
    d = d.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    return d[['date', LTG_COL]]


def _save_ltg_cache(path: Path, d: pd.DataFrame) -> None:
    tmp = path.with_suffix(path.suffix + '.tmp')
    out = d.copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce').dt.normalize()
    out[LTG_COL] = pd.to_numeric(out[LTG_COL], errors='coerce')
    out = out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    out.to_parquet(tmp, index=False)
    tmp.replace(path)



def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    if isinstance(x.columns, pd.MultiIndex):
        x.columns = [' | '.join([str(v) for v in tup if v is not None]).strip() for tup in x.columns]
    else:
        x.columns = [str(c).strip() for c in x.columns]
    return x

def _extract_ltg_history(raw: pd.DataFrame) -> pd.DataFrame:
    if raw is None or len(raw) == 0:
        return pd.DataFrame(columns=['date', LTG_COL])

    x = _flatten_columns(pd.DataFrame(raw).copy().reset_index())
    if x.empty:
        return pd.DataFrame(columns=['date', LTG_COL])

    # Resolve date column
    date_col = None
    for c in x.columns:
        cl = c.lower()
        if cl == 'date' or 'date' in cl:
            date_col = c
            break
    if date_col is None:
        date_col = x.columns[0]

    # Resolve value column
    id_like = set()
    for c in x.columns:
        cl = c.lower()
        if cl in {'instrument', 'ric', 'isin'} or 'instrument' in cl or cl.endswith('ric') or cl.endswith('isin'):
            id_like.add(c)

    value_cols = [c for c in x.columns if c != date_col and c not in id_like]

    val_col = None
    for c in value_cols:
        uc = c.upper().replace(' ', '')
        if 'TR.LTGMEAN' in uc:
            val_col = c
            break

    if val_col is None and len(value_cols) == 1:
        # Some responses rename the requested field label; keep single-value-column responses.
        val_col = value_cols[0]

    if val_col is None:
        for c in value_cols:
            sc = pd.to_numeric(x[c], errors='coerce')
            if int(sc.notna().sum()) > 0:
                val_col = c
                break

    if val_col is None:
        return pd.DataFrame(columns=['date', LTG_COL])

    out = pd.DataFrame({
        'date': pd.to_datetime(x[date_col], errors='coerce').dt.normalize(),
        LTG_COL: pd.to_numeric(x[val_col], errors='coerce'),
    })
    out = out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    return out[['date', LTG_COL]]


def _pull_ltg_history_segment(pull_id: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    if pd.isna(start) or pd.isna(end) or start > end:
        return pd.DataFrame(columns=['date', LTG_COL])

    # Try quarterly first, then monthly for denser fill if quarterly returns nothing.
    plans = ['quarterly', 'monthly']

    for interval in plans:
        last_err = None
        for r in range(LTG_MAX_RETRIES):
            try:
                raw = ld.get_history(
                    universe=[pull_id],
                    fields=[LTG_FIELD],
                    start=start.strftime('%Y-%m-%d'),
                    end=end.strftime('%Y-%m-%d'),
                    interval=interval,
                )
                out = _extract_ltg_history(raw)
                if not out.empty:
                    return out
                break
            except Exception as e:
                last_err = e
                err_txt = str(e)
                if 'Unable to resolve all requested identifiers' in err_txt:
                    # Permanent identifier resolution error -> try next candidate/interval without retries.
                    break
                time.sleep(LTG_BASE_SLEEP * (2 ** r) + random.random() * 0.3)

        if last_err is not None:
            continue

    return pd.DataFrame(columns=['date', LTG_COL])


def _build_company_candidates(company_req: pd.DataFrame) -> List[Tuple[str, str]]:
    """Collect all unique identifiers per company across time.

    Order: all ISINs, then all RIC_current, then all RIC (first-seen chronology).
    """
    q = company_req.copy().sort_values('date')

    out = []
    seen = set()

    def _norm_isin(value):
        if pd.isna(value):
            return ''
        v = str(value).strip()
        if not v:
            return ''
        if v.upper().startswith('ISIN:'):
            v = v.split(':', 1)[1].strip()
        return v

    def _add(id_type: str, value):
        if pd.isna(value):
            return
        v = str(value).strip()
        if not v:
            return
        key = (id_type, v)
        if key in seen:
            return
        seen.add(key)
        out.append(key)

    if 'RIC_current' in q.columns:
        for val in q['RIC_current']:
            _add('RIC', val)

    if 'RIC' in q.columns:
        for val in q['RIC']:
            _add('RIC', val)

    if 'ISIN' in q.columns:
        for val in q['ISIN']:
            isin = _norm_isin(val)
            if isin:
                _add('ISIN', isin)
                _add('ISIN', f'ISIN:{isin}')

    if 'pull_id' in q.columns and 'id_type' in q.columns:
        for id_type, pull_id in zip(q['id_type'], q['pull_id']):
            it = str(id_type).upper().strip() if pd.notna(id_type) else ''
            if it == 'ISIN':
                isin = _norm_isin(pull_id)
                if isin:
                    _add('ISIN', isin)
                    _add('ISIN', f'ISIN:{isin}')
            elif it == 'RIC':
                _add('RIC', pull_id)

    return out


def _update_company_ltg_cache(firm_id: str, id_type: str, pull_id: str, start: pd.Timestamp, end: pd.Timestamp, force_refresh: bool = False) -> pd.DataFrame:
    path = _cache_path_for_company_id(firm_id, id_type, pull_id)
    legacy_path = _cache_path_for_company_legacy(firm_id)

    if force_refresh:
        cached = pd.DataFrame(columns=['date', LTG_COL])
    else:
        cached_id = _load_ltg_cache(path)
        cached_legacy = _load_ltg_cache(legacy_path) if legacy_path.exists() else pd.DataFrame(columns=['date', LTG_COL])

        if cached_id.empty and cached_legacy.empty:
            cached = pd.DataFrame(columns=['date', LTG_COL])
        elif cached_id.empty:
            cached = cached_legacy.copy()
        elif cached_legacy.empty:
            cached = cached_id.copy()
        else:
            # Merge ID-specific and legacy cache; keep non-null LTG from either source.
            m = cached_id.merge(cached_legacy, on='date', how='outer', suffixes=('_id', '_legacy'))
            m[LTG_COL] = pd.to_numeric(m[f'{LTG_COL}_id'], errors='coerce').combine_first(
                pd.to_numeric(m[f'{LTG_COL}_legacy'], errors='coerce')
            )
            cached = m[['date', LTG_COL]].copy()
            cached = cached.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')

        # Persist merged view so future runs don't miss legacy coverage again.
        if not cached.empty:
            _save_ltg_cache(path, cached)

    segments = []
    if cached.empty:
        segments.append((start, end))
    else:
        cmin = cached['date'].min()
        cmax = cached['date'].max()

        # Hard short-circuit only if range is covered AND cache already has at least one real LTG value.
        _has_cached_values = bool(pd.to_numeric(cached[LTG_COL], errors='coerce').notna().any()) if LTG_COL in cached.columns else False
        if pd.notna(cmin) and pd.notna(cmax) and (cmin <= start) and (cmax >= end) and (not force_refresh) and _has_cached_values:
            return cached

        if LTG_CACHE_ONLY:
            return cached

        if not _has_cached_values:
            # Negative cache only -> retry full requested span.
            segments.append((start, end))
        else:
            if start < cmin:
                segments.append((start, cmin - pd.Timedelta(days=1)))
            if end > cmax:
                segments.append((cmax + pd.Timedelta(days=1), end))

    if LTG_CACHE_ONLY:
        return cached

    pulled_parts = []
    segment_marks = []
    for sdt, edt in segments:
        if sdt > edt:
            continue
        part = _pull_ltg_history_segment(pull_id, sdt, edt)
        if not part.empty:
            pulled_parts.append(part)
        # Mark queried span even if no values came back (negative cache).
        segment_marks.append(pd.DataFrame({'date': [sdt, edt], LTG_COL: [np.nan, np.nan]}))

    all_parts = []
    if not cached.empty:
        all_parts.append(cached)
    all_parts.extend(pulled_parts)
    all_parts.extend(segment_marks)

    if all_parts:
        all_df = pd.DataFrame.from_records(
            [rec for part in all_parts for rec in part.to_dict('records')],
            columns=['date', LTG_COL],
        )
    else:
        all_df = cached.copy()

    if (not all_df.empty) or force_refresh:
        _save_ltg_cache(path, all_df)

    # Return ID-specific cache if present; fallback to combined frame.
    out = _load_ltg_cache(path)
    return out if not out.empty else all_df


def _map_history_to_requests(req_dates: pd.Series, hist: pd.DataFrame, tol_days: int = 120) -> pd.DataFrame:
    left = pd.DataFrame({'date': pd.to_datetime(req_dates, errors='coerce').dt.normalize()}).dropna().sort_values('date')
    if left.empty:
        return pd.DataFrame(columns=['date', LTG_COL])

    if hist is None or hist.empty:
        left[LTG_COL] = np.nan
        return left

    right = hist[['date', LTG_COL]].copy()
    right['date'] = pd.to_datetime(right['date'], errors='coerce').dt.normalize()
    right[LTG_COL] = pd.to_numeric(right[LTG_COL], errors='coerce')
    right = right.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')

    out = pd.merge_asof(
        left,
        right,
        on='date',
        direction='backward',
        tolerance=pd.Timedelta(days=tol_days),
    )
    return out


# Build unique request rows (company x asof) with row-level identifiers.
req_cols = [c for c in ['firm_id', 'date', 'ISIN', 'RIC_current', 'RIC', 'id_type', 'pull_id', LTG_COL] if c in euro500_eps_ltg.columns]
req_all = (
    euro500_eps_ltg[req_cols]
    .dropna(subset=['firm_id', 'date'])
    .drop_duplicates(['firm_id', 'date'], keep='last')
    .reset_index(drop=True)
)

if req_all.empty:
    raise ValueError('No valid (firm_id, date) rows found for LTG pull.')

if FORCE_REFRESH_LTG or (not LTG_SKIP_FILLED) or (LTG_COL not in req_all.columns):
    req = req_all.copy()
else:
    pending_mask = pd.to_numeric(req_all[LTG_COL], errors='coerce').isna()
    req = req_all[pending_mask].copy()

company_candidates_map = {}
if not req.empty:
    for ck, g in req.groupby('firm_id', sort=False):
        company_candidates_map[str(ck)] = _build_company_candidates(g)

req['n_id_candidates'] = req['firm_id'].astype(str).map(lambda ck: len(company_candidates_map.get(ck, []))) if not req.empty else pd.Series(dtype='int64')

print('Step 4 request rows (company x asof):', len(req))
print('As-of range:', req['date'].min(), 'to', req['date'].max()) if not req.empty else print('As-of range: n/a (no pending rows)')
print('Unique companies:', req['firm_id'].nunique() if not req.empty else 0)
print('Active LSEG field:', {LTG_COL: LTG_FIELD})
print('Mode:', 'CACHE_ONLY' if LTG_CACHE_ONLY else 'CACHE+NETWORK')
print('ID fallback order (historical per firm): all ISINs -> all RIC_current -> all RIC (+ pull_id/id_type history)')

# Pull loop (batched + resume)
companies_all = req['firm_id'].dropna().astype(str).unique().tolist() if not req.empty else []
companies_total = len(companies_all)

existing_step_rows = pd.DataFrame(columns=['firm_id', 'date', LTG_COL, 'ltg_rank', 'ltg_id_type', 'ltg_pull_id'])
if LTG_STEP4_ROWS_READ_PATH.exists():
    try:
        existing_step_rows = pd.read_parquet(LTG_STEP4_ROWS_READ_PATH)
        if 'date' in existing_step_rows.columns:
            existing_step_rows['date'] = pd.to_datetime(existing_step_rows['date'], errors='coerce').dt.normalize()
        for c in [LTG_COL, 'ltg_rank', 'ltg_id_type', 'ltg_pull_id']:
            if c not in existing_step_rows.columns:
                existing_step_rows[c] = pd.NA if c != LTG_COL else np.nan
        existing_step_rows[LTG_COL] = pd.to_numeric(existing_step_rows[LTG_COL], errors='coerce')
        existing_step_rows = existing_step_rows[['firm_id', 'date', LTG_COL, 'ltg_rank', 'ltg_id_type', 'ltg_pull_id']].dropna(subset=['firm_id', 'date'])
        existing_step_rows = existing_step_rows.drop_duplicates(['firm_id', 'date'], keep='last')
    except Exception as e:
        print(f'Warning: failed to read Step4 rows cache, continuing empty: {e}')
        existing_step_rows = pd.DataFrame(columns=['firm_id', 'date', LTG_COL, 'ltg_rank', 'ltg_id_type', 'ltg_pull_id'])

processed_from_rows = set(existing_step_rows['firm_id'].dropna().astype(str).unique().tolist()) if not existing_step_rows.empty else set()
processed_from_ckpt = set()
if LTG_STEP4_CKPT_PATH.exists() or LTG_STEP4_CKPT_LEGACY_PATH.exists():
    try:
        ckpt_read_path = LTG_STEP4_CKPT_PATH if LTG_STEP4_CKPT_PATH.exists() else LTG_STEP4_CKPT_LEGACY_PATH
        ck = json.loads(ckpt_read_path.read_text())
        processed_from_ckpt = set(str(x) for x in ck.get('processed_companies', []) if str(x).strip())
    except Exception as e:
        print(f'Warning: failed to read Step4 checkpoint, ignoring: {e}')

cache_file_count = len(list(LTG_CACHE_DIR.glob('*.parquet')))
if cache_file_count == 0 and (processed_from_rows or processed_from_ckpt):
    print('Step4: cache directory is empty -> ignoring step rows/checkpoint and starting full rebuild.')
    processed_from_rows = set()
    processed_from_ckpt = set()

processed_companies = set(processed_from_rows) | set(processed_from_ckpt)
companies = [c for c in companies_all if str(c) not in processed_companies]
N = len(companies)

print('Resume info: total_companies=', companies_total, 'already_processed=', len(processed_companies), 'remaining=', N, 'cache_files=', cache_file_count)

run_t0 = time.time()
new_rows_out = []
bad_log_rows = []

total_cand_calls = 0
total_resolved = 0
total_unresolved = 0
total_item_found = {LTG_COL: 0}

if not LTG_CACHE_ONLY:
    ld.open_session()
try:
    if N == 0:
        print('No remaining companies to pull in Step 4.')

    n_batches = int(np.ceil(N / BATCH_SIZE_LTG)) if N > 0 else 0
    for b_ix, b_start in enumerate(range(0, N, BATCH_SIZE_LTG), start=1):
        b_end = min(N, b_start + BATCH_SIZE_LTG)
        batch_companies = companies[b_start:b_end]
        batch_new_rows = []
        batch_processed = []

        print(f'[BATCH {b_ix}/{n_batches}] companies={len(batch_companies)} idx={b_start+1}-{b_end}')

        for k, firm_id in enumerate(batch_companies, start=1):
            company_req = req[req['firm_id'] == firm_id].copy().sort_values('date')
            req_dates = company_req['date'].dropna().drop_duplicates().sort_values().reset_index(drop=True)
            if req_dates.empty:
                continue

            start = pd.to_datetime(req_dates.min()).normalize()
            end = pd.to_datetime(req_dates.max()).normalize()

            panel = pd.DataFrame({'date': req_dates})
            panel[LTG_COL] = np.nan
            panel['ltg_rank'] = pd.NA
            panel['ltg_id_type'] = pd.NA
            panel['ltg_pull_id'] = pd.NA

            cands = company_candidates_map.get(str(firm_id), [])
            cand_used = 0
            attempted_ids = []

            for rank, cand in enumerate(cands, start=1):
                if panel[LTG_COL].notna().all():
                    break
                if (not isinstance(cand, (list, tuple))) or len(cand) != 2:
                    continue

                cand_type = str(cand[0]).upper().strip()
                cand_id = str(cand[1]).strip()
                if not cand_type or not cand_id:
                    continue

                cand_used += 1
                total_cand_calls += 1
                attempted_ids.append(cand_id if str(cand_id).upper().startswith('ISIN:') else f'{cand_type}:{cand_id}')

                ltg_cache_path = _cache_path_for_company_id(str(firm_id), cand_type, cand_id)
                cached_pre = _load_ltg_cache(ltg_cache_path)
                _has_cache = False
                if cached_pre is not None and (not cached_pre.empty):
                    _cmin = cached_pre['date'].min()
                    _cmax = cached_pre['date'].max()
                    _has_val = bool(pd.to_numeric(cached_pre[LTG_COL], errors='coerce').notna().any())
                    _has_cache = bool(pd.notna(_cmin) and pd.notna(_cmax) and _cmin <= start and _cmax >= end and _has_val)

                if _has_cache and (not FORCE_REFRESH_LTG):
                    hist = cached_pre
                else:
                    hist = _update_company_ltg_cache(
                        firm_id=str(firm_id),
                        id_type=cand_type,
                        pull_id=cand_id,
                        start=start,
                        end=end,
                        force_refresh=FORCE_REFRESH_LTG,
                    )

                mapped = _map_history_to_requests(req_dates, hist, tol_days=LTG_ASOF_TOL_DAYS)
                panel = panel.merge(mapped.rename(columns={LTG_COL: f'{LTG_COL}_cand'}), on='date', how='left')

                fill_mask = panel[LTG_COL].isna() & panel[f'{LTG_COL}_cand'].notna()
                panel.loc[fill_mask, LTG_COL] = panel.loc[fill_mask, f'{LTG_COL}_cand']
                panel.loc[fill_mask, 'ltg_rank'] = rank
                panel.loc[fill_mask, 'ltg_id_type'] = cand_type
                panel.loc[fill_mask, 'ltg_pull_id'] = cand_id
                panel = panel.drop(columns=[f'{LTG_COL}_cand'])

            panel['firm_id'] = str(firm_id)
            panel = panel[['firm_id', 'date', LTG_COL, 'ltg_rank', 'ltg_id_type', 'ltg_pull_id']]

            resolved = int(panel[LTG_COL].notna().sum())
            unresolved = int(panel[LTG_COL].isna().sum())
            total_resolved += resolved
            total_unresolved += unresolved

            item_found = {LTG_COL: int(panel[LTG_COL].notna().sum())}
            total_item_found[LTG_COL] += item_found[LTG_COL]

            batch_new_rows.extend(panel.to_dict('records'))
            batch_processed.append(str(firm_id))

            miss = panel[panel[LTG_COL].isna()][['firm_id', 'date']].copy()
            if not miss.empty:
                miss['reason'] = 'no_data_after_fallback'
                miss['n_candidates'] = len(cands)
                bad_log_rows.extend(miss[['firm_id', 'date', 'reason', 'n_candidates']].to_dict('records'))

            tried_ids_unique = []
            _seen_ids = set()
            for _tid in attempted_ids:
                if _tid not in _seen_ids:
                    _seen_ids.add(_tid)
                    tried_ids_unique.append(_tid)

            resolved_dates = pd.to_datetime(panel.loc[panel[LTG_COL].notna(), 'date'], errors='coerce').dropna()
            if resolved_dates.empty:
                pulled_range = 'NA:NA'
            else:
                pulled_range = f"{resolved_dates.min().date()}:{resolved_dates.max().date()}"

            tried_ids_txt = ' | '.join(tried_ids_unique) if tried_ids_unique else 'NA'
            print(
                f'[BATCH {b_ix}/{n_batches}] [{b_start+k}/{N}] '
                f'firm_id={str(firm_id)} | cand_used={cand_used}/{len(cands)} | '
                f'resolved={resolved} | unresolved={unresolved} | found_{LTG_COL}={item_found.get(LTG_COL,0)} | '
                f'pulled_range={pulled_range} | tried_ids: {tried_ids_txt}'
            )

        if batch_new_rows:
            batch_df = pd.DataFrame(batch_new_rows)
            batch_df['date'] = pd.to_datetime(batch_df['date'], errors='coerce').dt.normalize()
            batch_df[LTG_COL] = pd.to_numeric(batch_df[LTG_COL], errors='coerce')
            batch_df = batch_df.dropna(subset=['firm_id', 'date'])

            if LTG_STEP4_ROWS_READ_PATH.exists():
                prev = pd.read_parquet(LTG_STEP4_ROWS_READ_PATH)
                if prev is None or prev.empty:
                    combined = batch_df.copy()
                else:
                    prev = prev.copy()
                    prev['date'] = pd.to_datetime(prev.get('date'), errors='coerce').dt.normalize()
                    for c in [LTG_COL, 'ltg_rank', 'ltg_id_type', 'ltg_pull_id']:
                        if c not in prev.columns:
                            prev[c] = pd.NA if c != LTG_COL else np.nan
                    prev[LTG_COL] = pd.to_numeric(prev[LTG_COL], errors='coerce')
                    prev = prev[['firm_id', 'date', LTG_COL, 'ltg_rank', 'ltg_id_type', 'ltg_pull_id']].dropna(subset=['firm_id', 'date'])
                    combined = pd.concat([prev, batch_df], ignore_index=True)
            else:
                combined = batch_df.copy()

            combined = combined.sort_values(['firm_id', 'date']).drop_duplicates(['firm_id', 'date'], keep='last')
            combined.to_parquet(LTG_STEP4_ROWS_PATH, index=False)
            new_rows_out = combined.to_dict('records')

        if batch_processed:
            processed_companies.update(batch_processed)
            ckpt_payload = {
                'processed_companies': sorted(processed_companies),
                'remaining_companies': max(0, companies_total - len(processed_companies)),
                'updated_at_utc': pd.Timestamp.utcnow().isoformat(),
                'rows': int(len(new_rows_out)),
            }
            LTG_STEP4_CKPT_PATH.write_text(json.dumps(ckpt_payload, ensure_ascii=False, indent=2))

        if BATCH_PAUSE_SEC_LTG > 0 and b_ix < n_batches:
            time.sleep(BATCH_PAUSE_SEC_LTG)

finally:
    try:
        if not LTG_CACHE_ONLY:
            ld.close_session()
    except Exception:
        pass

print(
    f'Done Step 4 (LTG): total_companies={companies_total}, run_remaining_start={N}, candidate_calls={total_cand_calls}, '
    f'resolved_rows={total_resolved}, unresolved_rows={total_unresolved}, found_{LTG_COL}={total_item_found.get(LTG_COL,0)}'
)

if LTG_STEP4_ROWS_READ_PATH.exists():
    ltg_panel = pd.read_parquet(LTG_STEP4_ROWS_READ_PATH)
else:
    ltg_panel = pd.DataFrame(new_rows_out)

if ltg_panel is None or ltg_panel.empty:
    ltg_panel = pd.DataFrame(columns=['firm_id', 'date', LTG_COL, 'ltg_rank', 'ltg_id_type', 'ltg_pull_id'])
else:
    ltg_panel = ltg_panel.copy()
    ltg_panel['date'] = pd.to_datetime(ltg_panel.get('date'), errors='coerce').dt.normalize()
    for c in [LTG_COL, 'ltg_rank', 'ltg_id_type', 'ltg_pull_id']:
        if c not in ltg_panel.columns:
            ltg_panel[c] = pd.NA if c != LTG_COL else np.nan
    ltg_panel[LTG_COL] = pd.to_numeric(ltg_panel[LTG_COL], errors='coerce')
    ltg_panel = ltg_panel[['firm_id', 'date', LTG_COL, 'ltg_rank', 'ltg_id_type', 'ltg_pull_id']]
    ltg_panel = ltg_panel.dropna(subset=['firm_id', 'date']).sort_values(['firm_id', 'date']).drop_duplicates(['firm_id', 'date'], keep='last').reset_index(drop=True)

ltg_panel.to_parquet(LTG_STEP4_ROWS_PATH, index=False)

ckpt_payload = {
    'processed_companies': sorted(set(str(x) for x in ltg_panel['firm_id'].dropna().astype(str).unique().tolist())),
    'remaining_companies': max(0, companies_total - int(ltg_panel['firm_id'].dropna().astype(str).nunique())),
    'updated_at_utc': pd.Timestamp.utcnow().isoformat(),
}
LTG_STEP4_CKPT_PATH.write_text(json.dumps(ckpt_payload, ensure_ascii=False, indent=2))

# Persist bad log (append-safe without pd.concat).
if bad_log_rows:
    bad_df = pd.DataFrame(bad_log_rows)
    if LTG_BAD_RICS_LOG.exists():
        old = pd.read_csv(LTG_BAD_RICS_LOG)
        for c in ['firm_id', 'date', 'reason', 'n_candidates']:
            if c not in old.columns:
                old[c] = pd.NA
            if c not in bad_df.columns:
                bad_df[c] = pd.NA
        out_bad = pd.DataFrame.from_records(
            old[['firm_id', 'date', 'reason', 'n_candidates']].to_dict('records')
            + bad_df[['firm_id', 'date', 'reason', 'n_candidates']].to_dict('records'),
            columns=['firm_id', 'date', 'reason', 'n_candidates'],
        )
    else:
        out_bad = bad_df

    out_bad['date'] = pd.to_datetime(out_bad['date'], errors='coerce').dt.normalize()
    out_bad = out_bad.drop_duplicates(subset=['firm_id', 'date', 'reason'], keep='last')
    out_bad.to_csv(LTG_BAD_RICS_LOG, index=False)

# Merge LTG back and keep diagnostics columns.
keep_diag_cols = ['ltg_rank', 'ltg_id_type', 'ltg_pull_id']
euro500_eps_ltg = euro500_eps_ltg.drop(columns=[c for c in keep_diag_cols if c in euro500_eps_ltg.columns], errors='ignore')

# Stage 1: preferred merge on firm_id + date
ltg_cols = ['firm_id', 'date', LTG_COL, 'ltg_rank', 'ltg_id_type', 'ltg_pull_id']
if 'firm_id' in euro500_eps_ltg.columns:
    ltg_panel_firm = ltg_panel[ltg_cols].rename(columns={'firm_id': 'firm_id'})
    euro500_eps_ltg = euro500_eps_ltg.merge(
        ltg_panel_firm,
        on=['firm_id', 'date'],
        how='left',
        suffixes=('', '_firm'),
    )

# Stage 2 fallback: merge on firm_id + date
add_ltg = euro500_eps_ltg.merge(
    ltg_panel[ltg_cols],
    on=['firm_id', 'date'],
    how='left',
    suffixes=('', '_key'),
)

old_ltg_series = pd.to_numeric(euro500_eps_ltg[LTG_COL], errors='coerce') if LTG_COL in euro500_eps_ltg.columns else pd.Series(np.nan, index=euro500_eps_ltg.index)
new_ltg_firm = pd.to_numeric(
    euro500_eps_ltg[f'{LTG_COL}_firm'] if f'{LTG_COL}_firm' in euro500_eps_ltg.columns else pd.Series(np.nan, index=euro500_eps_ltg.index),
    errors='coerce'
)
new_ltg_key = pd.to_numeric(
    add_ltg[f'{LTG_COL}_key'] if f'{LTG_COL}_key' in add_ltg.columns else pd.Series(np.nan, index=euro500_eps_ltg.index),
    errors='coerce'
)
euro500_eps_ltg[LTG_COL] = new_ltg_firm.combine_first(new_ltg_key).combine_first(old_ltg_series)

old_rank = pd.to_numeric(euro500_eps_ltg['ltg_rank'], errors='coerce') if 'ltg_rank' in euro500_eps_ltg.columns else pd.Series(np.nan, index=euro500_eps_ltg.index)
new_rank_firm = pd.to_numeric(
    euro500_eps_ltg['ltg_rank_firm'] if 'ltg_rank_firm' in euro500_eps_ltg.columns else pd.Series(np.nan, index=euro500_eps_ltg.index),
    errors='coerce'
)
new_rank_key = pd.to_numeric(
    add_ltg['ltg_rank_key'] if 'ltg_rank_key' in add_ltg.columns else pd.Series(np.nan, index=euro500_eps_ltg.index),
    errors='coerce'
)
euro500_eps_ltg['ltg_rank'] = new_rank_firm.combine_first(new_rank_key).combine_first(old_rank)

old_type = euro500_eps_ltg['ltg_id_type'] if 'ltg_id_type' in euro500_eps_ltg.columns else pd.Series(pd.NA, index=euro500_eps_ltg.index, dtype='string')
new_type_firm = euro500_eps_ltg.get('ltg_id_type_firm', pd.Series(pd.NA, index=euro500_eps_ltg.index, dtype='string'))
new_type_key = add_ltg.get('ltg_id_type_key', pd.Series(pd.NA, index=euro500_eps_ltg.index, dtype='string'))
euro500_eps_ltg['ltg_id_type'] = new_type_firm.fillna(new_type_key).fillna(old_type)

old_pull = euro500_eps_ltg['ltg_pull_id'] if 'ltg_pull_id' in euro500_eps_ltg.columns else pd.Series(pd.NA, index=euro500_eps_ltg.index, dtype='string')
new_pull_firm = euro500_eps_ltg.get('ltg_pull_id_firm', pd.Series(pd.NA, index=euro500_eps_ltg.index, dtype='string'))
new_pull_key = add_ltg.get('ltg_pull_id_key', pd.Series(pd.NA, index=euro500_eps_ltg.index, dtype='string'))
euro500_eps_ltg['ltg_pull_id'] = new_pull_firm.fillna(new_pull_key).fillna(old_pull)

for c in ['ltg_rank_firm', 'ltg_id_type_firm', 'ltg_pull_id_firm']:
    if c in euro500_eps_ltg.columns:
        euro500_eps_ltg = euro500_eps_ltg.drop(columns=[c])

# Save updated file.
drop_ltg_diag_cols = [c for c in ["ltg_rank", "ltg_id_type", "ltg_pull_id"] if c in euro500_eps_ltg.columns]
if drop_ltg_diag_cols:
    euro500_eps_ltg = euro500_eps_ltg.drop(columns=drop_ltg_diag_cols)

euro500_eps_ltg.to_parquet(EURO500_EPS_PATH, index=False)
print('Step4 rows saved:', LTG_STEP4_ROWS_PATH, 'rows:', len(ltg_panel))
print('Step4 checkpoint:', LTG_STEP4_CKPT_PATH)

# Coverage diagnostics: before vs after overall and explicitly 2015/2025.
post_ltg = pd.to_numeric(euro500_eps_ltg[LTG_COL], errors='coerce')
asof_year = pd.to_datetime(euro500_eps_ltg['date'], errors='coerce').dt.year

def _cov(mask: pd.Series, s: pd.Series) -> float:
    sub = s[mask]
    return float(sub.notna().mean()) if len(sub) > 0 else np.nan

rows = []
for yr in ['ALL', 2015, 2025]:
    if yr == 'ALL':
        m = pd.Series(True, index=euro500_eps_ltg.index)
    else:
        m = (asof_year == yr)
    rows.append({
        'year': yr,
        'rows': int(m.sum()),
        'coverage_before': _cov(m, prev_ltg),
        'coverage_after': _cov(m, post_ltg),
    })

coverage_cmp = pd.DataFrame(rows)
coverage_cmp['delta_pp'] = (coverage_cmp['coverage_after'] - coverage_cmp['coverage_before']) * 100.0

print('Saved:', EURO500_EPS_PATH)
print('Rows:', len(euro500_eps_ltg))
print('Unique firm_id:', euro500_eps_ltg['firm_id'].nunique(dropna=True))
print('Unique date:', euro500_eps_ltg['date'].nunique(dropna=True))
print('\nLTG coverage before vs after fallback:')
display(coverage_cmp)

if not ltg_panel.empty:
    by_id = (
        ltg_panel[ltg_panel['ltg_id_type'].notna()]
        .groupby('ltg_id_type', as_index=False)
        .agg(rows=('firm_id', 'size'))
        .sort_values('rows', ascending=False)
    )
    if not by_id.empty:
        by_id['share'] = by_id['rows'] / by_id['rows'].sum()
    print('Resolved LTG rows by winning identifier type:')
    display(by_id)
else:
    print('No LTG rows were resolved in this run.')


### 3A. Coverage-Analyse EPS und LTG

In [ ]:

if not EURO500_EPS_PATH.exists():
    raise FileNotFoundError(f'Missing file: {EURO500_EPS_PATH}')

cov = pd.read_parquet(EURO500_EPS_PATH).copy()

# Datumsanker fuer Jahresanalyse
if 'date' not in cov.columns:
    raise ValueError('Missing required date column for completeness analysis.')
cov['date'] = pd.to_datetime(cov['date'], errors='coerce').dt.normalize()

cov = cov[cov['date'].notna()].copy()
if cov.empty:
    raise ValueError('No valid date values found for completeness analysis.')

cov['year'] = cov['date'].dt.year

eps_cols = [c for c in ['EPS_FY1', 'EPS_FY2', 'EPS_FY3', 'EPS_FY4', 'EPS_FY5']]
for c in eps_cols + [LTG_COL]:
    if c not in cov.columns:
        cov[c] = np.nan
    cov[c] = pd.to_numeric(cov[c], errors='coerce')

cov['eps_any'] = cov[eps_cols].notna().any(axis=1)
cov['eps_all'] = cov[eps_cols].notna().all(axis=1)
cov['eps_partial'] = cov['eps_any'] & (~cov['eps_all'])
cov['eps_ltg_all'] = cov['eps_all'] & cov[LTG_COL].notna()
cov['eps_non_null_n'] = cov[eps_cols].notna().sum(axis=1)

agg = {'rows': ('year', 'size')}
for c in eps_cols + [LTG_COL]:
    agg[f'{c}_non_null'] = (c, lambda s: int(s.notna().sum()))
for c in ['eps_any', 'eps_all', 'eps_partial', 'eps_ltg_all']:
    agg[f'{c}_non_null'] = (c, lambda s: int(s.sum()))

by_year = cov.groupby('year', as_index=False).agg(**agg)

# Coverage-Raten ergaenzen
for c in eps_cols + [LTG_COL, 'eps_any', 'eps_all', 'eps_partial', 'eps_ltg_all']:
    by_year[f'{c}_coverage'] = by_year[f'{c}_non_null'] / by_year['rows']

# Lesbare Spaltenauswahl
show_cols = ['year', 'rows']
show_cols += [f'{c}_coverage' for c in eps_cols]
show_cols += [f'{LTG_COL}_coverage', 'eps_any_coverage', 'eps_all_coverage', 'eps_partial_coverage', 'eps_ltg_all_coverage']

print('Coverage by year (Step 3+4 outputs):')
display(by_year[show_cols].sort_values('year'))
print('Rows with partial EPS coverage (any but not all horizons):', int(cov['eps_partial'].sum()))

# Diagnostics: verify eps_any vs eps_all relationship
pattern = (
    cov.groupby('eps_non_null_n', as_index=False)
    .agg(rows=('eps_non_null_n', 'size'))
    .sort_values('eps_non_null_n')
)
pattern['share'] = pattern['rows'] / len(cov)
print('EPS non-null count distribution (0..5):')
display(pattern)
print('Rows with eps_any != eps_all:', int((cov['eps_any'] != cov['eps_all']).sum()))
print('Rows with eps_all=True and eps_any=False (should be 0):', int(((cov['eps_all']) & (~cov['eps_any'])).sum()))

# Schwachstellen: schlechteste Jahre
worst = (
    by_year[['year', 'eps_ltg_all_coverage']]
    .sort_values('eps_ltg_all_coverage', ascending=True)
    .head(10)
)
print('10 years with lowest joint EPS+LTG completeness:')
display(worst)

# Plot: Zeitverlauf der wichtigsten Coverage-Metriken
plot_cols = [
    'eps_any_coverage',
    'eps_all_coverage',
    'eps_partial_coverage',
    f'{LTG_COL}_coverage',
    'eps_ltg_all_coverage',
]

fig, ax = plt.subplots(figsize=(12, 5))
for col in plot_cols:
    ax.plot(by_year['year'], by_year[col], marker='o', label=col)

ax.set_title('Data completeness over years (after Step 3 + Step 4)')
ax.set_xlabel('year')
ax.set_ylabel('coverage share')
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


# Zusatzplot: Non-Financials only (trbc_sector != 'financials')
if 'trbc_sector' in cov.columns:
    cov_nonfin = cov[cov['trbc_sector'].astype(str).str.strip().str.lower() != 'financials'].copy()
    if not cov_nonfin.empty:
        agg_nf = {'rows': ('year', 'size')}
        for c in eps_cols + [LTG_COL]:
            agg_nf[f'{c}_non_null'] = (c, lambda s: int(s.notna().sum()))
        for c in ['eps_any', 'eps_all', 'eps_partial', 'eps_ltg_all']:
            agg_nf[f'{c}_non_null'] = (c, lambda s: int(s.sum()))

        by_year_nf = cov_nonfin.groupby('year', as_index=False).agg(**agg_nf)
        for c in eps_cols + [LTG_COL, 'eps_any', 'eps_all', 'eps_partial', 'eps_ltg_all']:
            by_year_nf[f'{c}_coverage'] = by_year_nf[f'{c}_non_null'] / by_year_nf['rows']

        fig, ax = plt.subplots(figsize=(12, 5))
        for col in plot_cols:
            ax.plot(by_year_nf['year'], by_year_nf[col], marker='o', label=col)

        ax.set_title('Data completeness over years (Step 3+4, Non-Financials only)')
        ax.set_xlabel('year')
        ax.set_ylabel('coverage share')
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print('No non-financial rows available for Non-Financials coverage plot.')
else:
    print('Column trbc_sector missing; skipping Non-Financials coverage plot.')







### 3B. LTG-Qualitaetscheck (Plausibilitaet)

In [ ]:
LTG_COL = 'TR.LTGMean'

if not EURO500_EPS_PATH.exists():
    raise FileNotFoundError(f'Missing file: {EURO500_EPS_PATH}')

ltg_df = pd.read_parquet(EURO500_EPS_PATH).copy()
if LTG_COL not in ltg_df.columns:
    raise KeyError(f'Column {LTG_COL} not found. Run Step 4 first.')

# Datentypen / Datumsfelder
for dc in ['date', 'date', 'formation_date', 'effective_date']:
    if dc in ltg_df.columns:
        ltg_df[dc] = pd.to_datetime(ltg_df[dc], errors='coerce')

ltg_df[LTG_COL] = pd.to_numeric(ltg_df[LTG_COL], errors='coerce')

if 'date' in ltg_df.columns and ltg_df['date'].notna().any():
    ltg_df['year'] = ltg_df['date'].dt.year
elif 'date' in ltg_df.columns:
    ltg_df['year'] = ltg_df['date'].dt.year
else:
    ltg_df['year'] = pd.NA

# 1) Overall coverage + basic stats
n_total = len(ltg_df)
n_non_null = int(ltg_df[LTG_COL].notna().sum())
n_missing = n_total - n_non_null
coverage = (n_non_null / n_total) if n_total else np.nan

outlier_lo = (ltg_df[LTG_COL] < -1).sum()
outlier_hi = (ltg_df[LTG_COL] > 1).sum()

summary = pd.DataFrame([{
    'rows_total': n_total,
    'ltg_non_null': n_non_null,
    'ltg_missing': n_missing,
    'coverage_share': coverage,
    'ltg_lt_-1_count': int(outlier_lo),
    'ltg_gt_+1_count': int(outlier_hi),
}])
display(summary)

ltg_ok = ltg_df[LTG_COL].dropna()
if len(ltg_ok) > 0:
    print('TR.LTGMean describe (raw):')
    display(ltg_ok.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

    p01, p99 = ltg_ok.quantile([0.01, 0.99])
    ltg_w = ltg_ok.clip(lower=p01, upper=p99)
    print('TR.LTGMean describe (winsorized 1%-99%):')
    display(ltg_w.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(ltg_ok, bins=80)
    ax[0].set_title('TR.LTGMean distribution (raw)')
    ax[0].set_xlabel(LTG_COL)

    ax[1].hist(ltg_w, bins=80)
    ax[1].set_title('TR.LTGMean distribution (winsor 1%-99%)')
    ax[1].set_xlabel(LTG_COL)
    plt.tight_layout()
    plt.show()


    # Zusatzplot: LTG coverage by year (Non-Financials only)
    if 'trbc_sector' in ltg_df.columns:
        ltg_nonfin = ltg_df[ltg_df['trbc_sector'].astype(str).str.strip().str.lower() != 'financials'].copy()
        if ltg_nonfin['year'].notna().any():
            cov_year_nf = ltg_nonfin.groupby('year', as_index=False).agg(
                rows=(LTG_COL, 'size'),
                non_null=(LTG_COL, lambda s: s.notna().sum()),
                median=(LTG_COL, 'median'),
                p10=(LTG_COL, lambda s: s.quantile(0.10)),
                p90=(LTG_COL, lambda s: s.quantile(0.90)),
            )
            cov_year_nf['coverage'] = cov_year_nf['non_null'] / cov_year_nf['rows']

            fig, ax = plt.subplots(1, 2, figsize=(12, 4))
            ax[0].plot(cov_year_nf['year'], cov_year_nf['coverage'])
            ax[0].set_title('LTG coverage by year (Non-Financials)')
            ax[0].set_xlabel('year')
            ax[0].set_ylabel('coverage')
            ax[0].set_ylim(0, 1.02)
            ax[0].grid(alpha=0.3)

            ax[1].plot(cov_year_nf['year'], cov_year_nf['median'], label='median')
            ax[1].plot(cov_year_nf['year'], cov_year_nf['p10'], label='p10', alpha=0.7)
            ax[1].plot(cov_year_nf['year'], cov_year_nf['p90'], label='p90', alpha=0.7)
            ax[1].set_title('LTG level by year (Non-Financials)')
            ax[1].set_xlabel('year')
            ax[1].legend()
            ax[1].grid(alpha=0.3)
            plt.tight_layout()
            plt.show()
        else:
            print('No valid year info for LTG Non-Financials coverage plot.')
    else:
        print('Column trbc_sector missing; skipping LTG Non-Financials coverage plot.')


else:
    print('No non-null TR.LTGMean values found.')

# 2) Coverage over time
if 'year' in ltg_df.columns and ltg_df['year'].notna().any():
    cov_year = ltg_df.groupby('year', as_index=False).agg(
        rows=(LTG_COL, 'size'),
        non_null=(LTG_COL, lambda s: s.notna().sum()),
        median=(LTG_COL, 'median'),
        p10=(LTG_COL, lambda s: s.quantile(0.10)),
        p90=(LTG_COL, lambda s: s.quantile(0.90)),
    )
    cov_year['coverage'] = cov_year['non_null'] / cov_year['rows']
    display(cov_year.head(20))

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(cov_year['year'], cov_year['coverage'])
    ax[0].set_title('LTG coverage by year')
    ax[0].set_xlabel('year')
    ax[0].set_ylabel('coverage')
    ax[0].grid(alpha=0.3)

    ax[1].plot(cov_year['year'], cov_year['median'], label='median')
    ax[1].plot(cov_year['year'], cov_year['p10'], label='p10', alpha=0.7)
    ax[1].plot(cov_year['year'], cov_year['p90'], label='p90', alpha=0.7)
    ax[1].set_title('LTG level by year')
    ax[1].set_xlabel('year')
    ax[1].legend()
    ax[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()



    # Zusatzplot: LTG coverage by year (Non-Financials, coverage section)
    if 'trbc_sector' in ltg_df.columns:
        ltg_nonfin_cov = ltg_df[ltg_df['trbc_sector'].astype(str).str.strip().str.lower() != 'financials'].copy()
        if ltg_nonfin_cov['year'].notna().any():
            cov_year_nf = ltg_nonfin_cov.groupby('year', as_index=False).agg(
                rows=(LTG_COL, 'size'),
                non_null=(LTG_COL, lambda s: s.notna().sum()),
                median=(LTG_COL, 'median'),
                p10=(LTG_COL, lambda s: s.quantile(0.10)),
                p90=(LTG_COL, lambda s: s.quantile(0.90)),
            )
            cov_year_nf['coverage'] = cov_year_nf['non_null'] / cov_year_nf['rows']

            fig, ax = plt.subplots(1, 2, figsize=(12, 4))
            ax[0].plot(cov_year_nf['year'], cov_year_nf['coverage'])
            ax[0].set_title('LTG coverage by year (Non-Financials)')
            ax[0].set_xlabel('year')
            ax[0].set_ylabel('coverage')
            ax[0].set_ylim(0, 1.02)
            ax[0].grid(alpha=0.3)

            ax[1].plot(cov_year_nf['year'], cov_year_nf['median'], label='median')
            ax[1].plot(cov_year_nf['year'], cov_year_nf['p10'], label='p10', alpha=0.7)
            ax[1].plot(cov_year_nf['year'], cov_year_nf['p90'], label='p90', alpha=0.7)
            ax[1].set_title('LTG level by year (Non-Financials)')
            ax[1].set_xlabel('year')
            ax[1].legend()
            ax[1].grid(alpha=0.3)
            plt.tight_layout()
            plt.show()
        else:
            print('No valid year info for LTG Non-Financials coverage plot.')
    else:
        print('Column trbc_sector missing; skipping LTG Non-Financials coverage plot.')

# 3) Cross-section by sector / country
for grp_col in ['trbc_sector', 'hq_country']:
    if grp_col in ltg_df.columns:
        g = (
            ltg_df.groupby(grp_col, dropna=False)[LTG_COL]
            .agg(n='size', non_null=lambda s: s.notna().sum(), median='median', mean='mean')
            .reset_index()
        )
        g['coverage'] = g['non_null'] / g['n']
        g = g.sort_values(['coverage', 'n'], ascending=[False, False])
        print(f'By {grp_col}:')
        display(g.head(20))


